In [0]:
from pyspark.sql.functions import col

silver = spark.table("nyc311_project.silver.service_requests")
quarantine = spark.table("nyc311_project.quarantine.service_requests_unlocatable")
gold_dim = spark.table("nyc311_project.gold.dim_complaint_status")
gold_fact = spark.table("nyc311_project.gold.fact_requests")

print("=" * 60)
print("1. ROW COUNT RECONCILIATION")
print("=" * 60)
bronze_count = spark.table("nyc311_project.bronze.service_requests").count()
silver_count = silver.count()
quarantine_count = quarantine.count()
print(f"Bronze: {bronze_count}")
print(f"Silver (good): {silver_count}")
print(f"Quarantine: {quarantine_count}")
print(f"Silver + Quarantine = {silver_count + quarantine_count} (should equal Bronze: {bronze_count})")

1. ROW COUNT RECONCILIATION
Bronze: 200000
Silver (good): 199053
Quarantine: 947
Silver + Quarantine = 200000 (should equal Bronze: 200000)


In [0]:
print("=" * 60)
print("2. NULL CHECKS ON KEY COLUMNS")
print("=" * 60)
for c in ["unique_key", "agency", "complaint_type", "created_ts"]:
    nulls = silver.filter(col(c).isNull()).count()
    print(f"  {c}: {nulls} nulls")

2. NULL CHECKS ON KEY COLUMNS
  unique_key: 0 nulls
  agency: 0 nulls
  complaint_type: 0 nulls
  created_ts: 0 nulls


In [0]:
print("=" * 60)
print("3. DUPLICATE KEY CHECK")
print("=" * 60)
total = silver.count()
distinct = silver.select("unique_key").distinct().count()
print(f"Silver: {total} rows, {distinct} distinct unique_key ({'OK' if total == distinct else 'DUPLICATES FOUND'})")

3. DUPLICATE KEY CHECK
Silver: 199053 rows, 199053 distinct unique_key (OK)


In [0]:
print("=" * 60)
print("4. SCD2 INTEGRITY CHECK — every key should have exactly ONE current row")
print("=" * 60)
from pyspark.sql.functions import count as _count

current_counts = (gold_dim.filter(col("is_current") == True)
    .groupBy("unique_key")
    .agg(_count("*").alias("current_row_count"))
    .filter(col("current_row_count") != 1))

bad_keys = current_counts.count()
print(f"Keys with != 1 current row: {bad_keys} (should be 0)")

4. SCD2 INTEGRITY CHECK — every key should have exactly ONE current row
Keys with != 1 current row: 0 (should be 0)


In [0]:
print("=" * 60)
print("5. REFERENTIAL INTEGRITY — fact table vs dimension")
print("=" * 60)
orphans = gold_fact.join(gold_dim.filter(col("is_current") == True), "unique_key", "left_anti")
print(f"Fact rows with no matching current dimension row: {orphans.count()} (should be 0)")

5. REFERENTIAL INTEGRITY — fact table vs dimension
Fact rows with no matching current dimension row: 0 (should be 0)


In [0]:
print("=" * 60)
print("6. CASING STANDARDIZATION CHECK")
print("=" * 60)
# Confirm no more ALL-CAPS values slipped through in complaint_type
all_caps_check = silver.filter(col("complaint_type") == col("complaint_type").cast("string")).select("complaint_type").distinct()
sample = [row.complaint_type for row in silver.select("complaint_type").distinct().limit(10).collect()]
print("Sample complaint_type values:", sample)

6. CASING STANDARDIZATION CHECK
Sample complaint_type values: ['Blocked Driveway', 'Noise - Residential', 'Noise - Commercial', 'Illegal Parking', 'Noise - Street/sidewalk', 'Disorderly Youth', 'Illegal Fireworks', 'Street Light Condition', 'Food Establishment', 'Unsanitary Condition']


In [0]:
from pyspark.sql.functions import col

fact_keys = spark.table("nyc311_project.gold.fact_requests").select("unique_key")
dim_keys = spark.table("nyc311_project.gold.dim_complaint_status").filter(col("is_current") == True).select("unique_key")

missing = fact_keys.join(dim_keys, "unique_key", "left_anti")
print(f"Missing keys count: {missing.count()}")
display(missing.limit(10))

# Check: do these keys exist in dim_complaint_status AT ALL (even as non-current)?
dim_all_keys = spark.table("nyc311_project.gold.dim_complaint_status").select("unique_key").distinct()
truly_absent = missing.join(dim_all_keys, "unique_key", "left_anti")
print(f"Keys completely absent from dim table (not even as old version): {truly_absent.count()}")

Missing keys count: 1517


unique_key
61664866
61655274
61654509
61655240
61658383
61659267
61664766
61659400
61658342
61658316


Keys completely absent from dim table (not even as old version): 1517


In [0]:
from pyspark.sql.functions import col
orphans = spark.table("nyc311_project.gold.fact_requests") \
    .join(spark.table("nyc311_project.gold.dim_complaint_status").filter(col("is_current") == True), "unique_key", "left_anti")
print(f"Orphans: {orphans.count()} (should still be 0)")

Orphans: 0 (should still be 0)
